# Wind Turbine Load Estimation: RAG Pipeline

This notebook orchestrates a local, confidential Retrieval-Augmented Generation (RAG) pipeline. It ingests technical research papers, stores them in a local Chroma vector database, and uses a Groq-powered LLM with a strict prompt template to extract specific engineering insights.

In [6]:
import os
import sys
from dotenv import load_dotenv

# Append the src directory to the system path to import our custom modules
sys.path.append(os.path.abspath('../src'))

# Load environment variables (Ensure GROQ_API_KEY is set in a .env file in the root directory)
load_dotenv()

True

## 1. Data Ingestion
Load the research PDF and slice it into overlapping 1000-character chunks.

In [ ]:
from data import DocumentProcessor

# Initialize processor pointing to your specific research document
processor = DocumentProcessor(file_path="../data/torque_research_paper.pdf")
document_chunks = processor.process()

print(f"Successfully generated {len(document_chunks)} document chunks.")

FileNotFoundError: Input document does not exist. Please check. - ../data/Evaluation_of_the_Robustness_of_Machine_Learning.pdf

## 2. Vector Storage & Retrieval
Embed the chunks using HuggingFace and store them locally. Configure a Maximum Marginal Relevance (MMR) retriever to fetch diverse context.

In [ ]:
from vector_store import VectorStoreManager

# Initialize the manager and build the database
vector_manager = VectorStoreManager(persist_dir="../data/chroma_db")
vector_db = vector_manager.build_database(document_chunks)

# Retrieve the configured MMR search engine
retriever = vector_manager.get_retriever(vector_db)

print("ChromaDB built and MMR retriever configured.")

ChromaDB built and MMR retriever configured.


## 3. LLM Generation Chain
Construct the LangChain Expression Language (LCEL) pipeline enforcing a strict, zero-hallucination prompt.

In [ ]:
from model import GenerationPipeline

# Initialize the pipeline and build the chain
pipeline = GenerationPipeline(retriever=retriever)
rag_chain = pipeline.build_chain()

print("LCEL Generation chain successfully compiled.")

LCEL Generation chain successfully compiled.


## 4. Execution
Query the pipeline regarding surrogate emulators and structural load evaluations.

In [ ]:
# Execute a domain-specific query
query = "How do machine learning surrogate models improve the evaluation of Ultimate Limit State (ULS) and Fatigue Limit State (FLS) loads in wind turbines compared to traditional aeroelastic simulations?"

# Invoke the LCEL chain
final_answer = rag_chain.invoke(query)

print("### Query Output ###\n")
print(final_answer)

### Query Output ###

Machine‑learning surrogate models accelerate and broaden the assessment of wind‑turbine limit states by learning the relationship between SCADA‑measured operational signals and the corresponding fatigue loads that are normally obtained from expensive aero‑elastic simulations. Once trained, these models can predict Ultimate Limit State (ULS) and Fatigue Limit State (FLS) loads in real time or near‑real time, using only live SCADA data. This contrasts with traditional aero‑elastic tools (e.g., FAST, BHAWC), which require detailed, computationally intensive simulations that are typically run offline and cannot be updated continuously. Consequently, ML surrogates enable more frequent, data‑driven monitoring of ULS and FLS loads, potentially improving safety assessment and operational decision‑making.
